# Phase 4 — Separating the interface from scale

Two runs that answer the central criticisms in the review, both resumable and
saved to Drive.

| Task | Cost |
|---|---|
| `generative` | ~2-4 h |
| `seeds10` | ~1 h |


---
## 1. Drive and resumption

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
import os, sys, re, json, time, gc, shutil, hashlib, warnings
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Dict, List, Optional, Tuple
import numpy as np, pandas as pd, torch, torch.nn as nn, torch.nn.functional as F
warnings.filterwarnings("ignore")
try:
    from transformers.utils import logging as hl; hl.set_verbosity_error()
except Exception: pass

DRIVE = Path("/content/drive/MyDrive/anonymed"); DRIVE.mkdir(parents=True, exist_ok=True)

@dataclass
class Config:
    granularity: str = "sub"
    teacher: str = "Qwen/Qwen2.5-7B-Instruct"
    teacher_dtype: str = "bfloat16"
    gen_batch: int = 8
    gen_max_new: int = 1400          # generous headroom for rewriting a 150-word window
    student: str = "neuralmind/bert-base-portuguese-cased"
    max_length: int = 512
    batch_size: int = 16
    lr: float = 5e-5
    epochs: int = 5
    kd_T: float = 4.0
    kd_alpha: float = 0.95
    seeds10: Tuple[int, ...] = (13, 21, 42, 87, 101, 7, 55, 123, 202, 314)

CFG = Config()
RUN = DRIVE / f"phase4_{CFG.granularity}"; RUN.mkdir(parents=True, exist_ok=True)
(RUN/"preds").mkdir(exist_ok=True)
P2, P3 = DRIVE/f"phase2_{CFG.granularity}", DRIVE/f"phase3_{CFG.granularity}"

_T0 = time.time()
def log(m):
    line = f"[{time.strftime('%H:%M:%S')} +{(time.time()-_T0)/60:5.1f} min] {m}"
    print(line); open(RUN/"run.log","a",encoding="utf-8").write(line+"\n")
def save(t,p): (RUN/f"{t}.json").write_text(json.dumps(p,indent=2,default=float),encoding="utf-8"); log(f"saved: {t}")
def load(t):
    f=RUN/f"{t}.json"; return json.loads(f.read_text(encoding="utf-8")) if f.exists() else None
def task(name, fn, force=False):
    c = None if force else load(name)
    if c is not None: log(f"skip: {name} (already done)"); return c
    log(f"start: {name}"); out = fn(); save(name, out); return out

if torch.cuda.is_available():
    log(f"gpu {torch.cuda.get_device_name(0)}")
log(f"run dir {RUN}")

[14:34:22 +  0.0 min] gpu NVIDIA A100-SXM4-80GB
[14:34:22 +  0.0 min] run dir /content/drive/MyDrive/anonymed/phase4_sub


---
## 2. Data and labels

In [ ]:
SUB = ["AGE","PHONE","EMAIL","DATE","IDNUM","MEDICAL_RECORD","HEALTH_PLAN","STREET","CITY",
       "ZIP","STATE","COUNTRY","LOCATION_OTHER","ORGANIZATION","HOSPITAL","PATIENT","DOCTOR",
       "PROFESSION","OTHER"]
LABELS = ["O"] + [f"{p}-{t}" for t in SUB for p in ("B","I")]
LABEL2ID = {l:i for i,l in enumerate(LABELS)}; ID2LABEL = {i:l for l,i in LABEL2ID.items()}
N_LABELS = len(LABELS)

def _find(n):
    for d in (DRIVE, Path("/content"), Path("/content/outputs"), Path(".")):
        if (d/n).exists(): return d/n
    raise FileNotFoundError(f"{n} not found -- run the earlier phases first")

def load_windows(stem):
    d = json.loads(_find(f"{stem}_{CFG.granularity}.json").read_text())
    for s in d.values():
        for c in s: c["tags"] = [ID2LABEL[i] for i in c["label_ids"]]
    return d

CORPUS      = load_windows("train_windows")
CORPUS_EVAL = load_windows("eval_windows")
SOFT = {}
for f in sorted((P2/"soft").glob("shard_*.npz")):
    z = np.load(f); SOFT.update({k: z[k] for k in z.files})
log(f"soft labels: {len(SOFT)} windows")
print("test windows:", len(CORPUS_EVAL["test"]))

[14:34:51 +  0.5 min] soft labels: 2241 janelas
test windows: 695


---
## 3. Metrics and training (identical to the earlier phases)

In [ ]:
!pip install -q transformers seqeval

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 4.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done


In [ ]:
from torch.utils.data import Dataset
from transformers import (AutoTokenizer, AutoModelForTokenClassification, AutoModelForCausalLM,
                          TrainingArguments, Trainer, default_data_collator)
from seqeval.metrics import classification_report as seq_report
from seqeval.scheme import IOB2

student_tok = AutoTokenizer.from_pretrained(CFG.student)

class NERSet(Dataset):
    def __init__(self, chunks, soft=None): self.chunks, self.soft = chunks, soft
    def __len__(self): return len(self.chunks)
    def __getitem__(self, i):
        c = self.chunks[i]
        enc = student_tok(c["tokens"], is_split_into_words=True, truncation=True,
                          max_length=CFG.max_length, padding="max_length", return_tensors="pt")
        wids = enc.word_ids(0)
        labels = torch.full((CFG.max_length,), -100, dtype=torch.long)
        soft = torch.zeros(CFG.max_length, N_LABELS) if self.soft is not None else None
        prev = None
        for pos, wid in enumerate(wids):
            if wid is None or wid == prev: prev = wid; continue
            labels[pos] = c["label_ids"][wid]
            if soft is not None: soft[pos] = torch.from_numpy(self.soft[c["id"]][wid].astype(np.float32))
            prev = wid
        it = {"input_ids": enc.input_ids[0], "attention_mask": enc.attention_mask[0], "labels": labels}
        if soft is not None: it["soft_logits"] = soft
        return it

class KDTrainer(Trainer):
    def __init__(self,*a,kd_T=4.0,kd_alpha=0.95,**kw):
        super().__init__(*a,**kw); self.kd_T, self.kd_alpha = kd_T, kd_alpha
    def compute_loss(self, model, inputs, return_outputs=False, **kw):
        soft = inputs.pop("soft_logits", None); labels = inputs["labels"]
        out = model(input_ids=inputs["input_ids"], attention_mask=inputs["attention_mask"])
        ce = F.cross_entropy(out.logits.view(-1,N_LABELS), labels.view(-1), ignore_index=-100)
        if soft is None: return (ce,out) if return_outputs else ce
        m = labels.view(-1)!=-100; T=self.kd_T
        s = F.log_softmax(out.logits.view(-1,N_LABELS)[m]/T,-1)
        t = F.softmax(soft.view(-1,N_LABELS)[m]/T,-1)
        loss = self.kd_alpha*ce + (1-self.kd_alpha)*(T**2)*F.kl_div(s,t,reduction="batchmean")
        return (loss,out) if return_outputs else loss

def metrics(yt, yp):
    r = seq_report(yt, yp, output_dict=True, mode="strict", scheme=IOB2, zero_division=0)
    return {"macro_f1": r["macro avg"]["f1-score"], "micro_f1": r["micro avg"]["f1-score"],
            "micro_precision": r["micro avg"]["precision"], "micro_recall": r["micro avg"]["recall"],
            "per_type": {k:v for k,v in r.items() if not k.endswith("avg")}}
slim = lambda m: {k:v for k,v in m.items() if k!="per_type"}

@torch.no_grad()
def predict(model, chunks, device=None, bs=32):
    device = device or ("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device).eval(); ds = NERSet(chunks); yt, yp = [], []
    for i in range(0, len(ds), bs):
        items=[ds[j] for j in range(i,min(i+bs,len(ds)))]
        ii=torch.stack([x["input_ids"] for x in items]).to(device)
        am=torch.stack([x["attention_mask"] for x in items]).to(device)
        lb=torch.stack([x["labels"] for x in items])
        pr=model(input_ids=ii,attention_mask=am).logits.argmax(-1).cpu()
        for b in range(len(items)):
            m=lb[b]!=-100
            yt.append([ID2LABEL[int(v)] for v in lb[b][m]])
            yp.append([ID2LABEL[int(v)] for v in pr[b][m]])
    return yt, yp

def set_seed(s):
    import random; random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
    torch.backends.cudnn.deterministic=True

def train_one(seed, soft=None, tag="r"):
    set_seed(seed)
    m = AutoModelForTokenClassification.from_pretrained(CFG.student, num_labels=N_LABELS,
        id2label=ID2LABEL, label2id=LABEL2ID)
    args = TrainingArguments(output_dir=f"/content/t_{tag}", report_to="none", learning_rate=CFG.lr,
        num_train_epochs=CFG.epochs, per_device_train_batch_size=CFG.batch_size, warmup_ratio=0.1,
        weight_decay=0.01, lr_scheduler_type="cosine", seed=seed, data_seed=seed,
        save_strategy="no", logging_strategy="no", bf16=torch.cuda.is_available(),
        disable_tqdm=True, remove_unused_columns=False)
    KDTrainer(model=m, args=args, train_dataset=NERSet(CORPUS["train"], soft),
              kd_T=CFG.kd_T, kd_alpha=CFG.kd_alpha, data_collator=default_data_collator).train()
    return m
free = lambda m: (m.cpu(), gc.collect(), torch.cuda.empty_cache())
print("components ready")

config.json:   0%|          | 0.00/647 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/43.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/210k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

componentes prontos


---
## 4. Task A — Qwen in the generative formulation



In [ ]:
TAG_RE = re.compile(r"<([A-Z_]+)>(.*?)</\1/>", re.S)

def gen_fewshot(n=3, max_words=40):
    """The same examples used for the constrained run, rendered as tagged text."""
    out = []
    for c in CORPUS["train"]:
        tg, tk = c["tags"][:max_words], c["tokens"][:max_words]
        if sum(t.startswith("B-") for t in tg) >= 3 and sum(t.startswith("I-") for t in tg) >= 2:
            plain, tagged, i = [], [], 0
            while i < len(tk):
                if tg[i].startswith("B-"):
                    typ = tg[i][2:]; j = i+1
                    while j < len(tk) and tg[j] == f"I-{typ}": j += 1
                    for k in range(i, j):
                        tagged.append(f"<{typ}>{tk[k]}</{typ}/>")
                    plain += tk[i:j]; i = j
                else:
                    tagged.append(tk[i]); plain.append(tk[i]); i += 1
            out.append((" ".join(plain), " ".join(tagged)))
        if len(out) >= n: break
    return out

def gen_prompt(tok, text, shots):
    """Build the generative rewriting prompt.

    The instructions stay in Portuguese on purpose: they are the experimental
    instrument, matched to the constrained run and to the protocol of Schiezaro
    et al., and the corpus is Brazilian Portuguese clinical text. Translating
    them would change what is being measured.
    """
    legend = ", ".join(SUB)
    ex = "\n\n".join(f"Entrada:\n{a}\n\nSaída:\n{b}" for a, b in shots)
    sysm = ("Você anonimiza prontuários médicos brasileiros. Reescreva o texto de entrada "
            "EXATAMENTE como recebido, inserindo tags em torno de cada palavra que contenha "
            "informação pessoal identificável.\n\n"
            f"TIPOS DE ENTIDADE: {legend}\n\n"
            "FORMATO: <TIPO>palavra</TIPO/> — uma tag por palavra. Entidades de várias "
            "palavras recebem uma tag por palavra.\n\n"
            "REGRAS:\n"
            "- Reproduza todas as palavras do texto original, na mesma ordem.\n"
            "- Não adicione, remova nem reescreva palavras.\n"
            "- Não produza nenhum texto além do prontuário reescrito.\n\n"
            f"EXEMPLOS:\n{ex}")
    return tok.apply_chat_template(
        [{"role":"system","content":sysm},{"role":"user","content":"Entrada:\n"+text+"\n\nSaída:"}],
        tokenize=False, add_generation_prompt=True)

def align_generated(gen_text, gold_tokens):
    """Project the model's tagged rewrite back onto the gold token sequence.

    Words the model omitted, reordered beyond recovery or altered receive O.
    This is conservative: it can only lower the generative score, never raise it.
    """
    pairs = []           # (word, type|None) in generation order
    pos = 0
    for m in TAG_RE.finditer(gen_text):
        for w in gen_text[pos:m.start()].split():
            pairs.append((w, None))
        for w in m.group(2).split():
            pairs.append((w, m.group(1)))
        pos = m.end()
    for w in gen_text[pos:].split():
        pairs.append((w, None))

    tags, gi = [], 0
    for w_gold in gold_tokens:
        typ, look = None, gi
        while look < min(gi + 8, len(pairs)):          # small forward window
            if pairs[look][0] == w_gold:
                typ = pairs[look][1]; gi = look + 1; break
            look += 1
        tags.append(typ)
    out, prev = [], None
    for t in tags:
        if t is None or f"B-{t}" not in LABEL2ID:
            out.append("O"); prev = None
        else:
            out.append(f"{'I' if t == prev else 'B'}-{t}"); prev = t
    return out

def run_generative():
    tok = AutoTokenizer.from_pretrained(CFG.teacher, padding_side="left")
    model = AutoModelForCausalLM.from_pretrained(CFG.teacher,
        torch_dtype=getattr(torch, CFG.teacher_dtype), device_map="auto").eval()
    shots = gen_fewshot()
    chunks = CORPUS_EVAL["test"]
    yt, yp, raw = [], [], []
    t0 = time.time()
    with torch.no_grad():
        for st in range(0, len(chunks), CFG.gen_batch):
            batch = chunks[st:st+CFG.gen_batch]
            prompts = [gen_prompt(tok, " ".join(c["tokens"]), shots) for c in batch]
            enc = tok(prompts, return_tensors="pt", padding=True).to(model.device)
            out = model.generate(**enc, max_new_tokens=CFG.gen_max_new, do_sample=False,
                                 temperature=None, top_p=None,
                                 pad_token_id=tok.pad_token_id or tok.eos_token_id)
            for b, c in enumerate(batch):
                gen = tok.decode(out[b][enc.input_ids.shape[1]:], skip_special_tokens=True)
                yt.append(c["tags"]); yp.append(align_generated(gen, c["tokens"]))
                if len(raw) < 3: raw.append(gen[:1200])
            if (st // CFG.gen_batch) % 10 == 0:
                el = time.time()-t0; d = st+len(batch)
                log(f"  generative {d}/{len(chunks)}  ETA {el/max(d,1)*(len(chunks)-d)/60:.0f} min")
    del model; gc.collect(); torch.cuda.empty_cache()
    (RUN/"preds"/"qwen_generative.json").write_text(json.dumps(yp))
    m = metrics(yt, yp)
    return {**slim(m), "per_type": m["per_type"], "raw_samples": raw,
            "coverage": float(np.mean([sum(t!="O" for t in p)>0 for p in yp]))}

GEN = task("generative", run_generative)
print("\n--- sample of raw teacher output ---")
print(GEN["raw_samples"][0][:700] if GEN.get("raw_samples") else "(old cache, no samples stored)")
print(f"\nQwen2.5-7B GENERATIVE    macro {GEN['macro_f1']:.4f}   micro {GEN['micro_f1']:.4f}"
      f"   recall {GEN['micro_recall']:.4f}")
print(f"Qwen2.5-7B CONSTRAINED   macro 0.1886   micro 0.3244   recall 0.2836   (phase 2)")
_d = 100*(GEN['micro_f1'] - 0.3244)
print(f"\ninterface gap, same model: {_d:+.2f} pp micro F1")
if _d > 20:
    print("=> The interface is the bottleneck. Title and thesis rest on direct")
    print("   evidence, with scale controlled.")
elif _d < 5:
    print("=> Scale explains most of it. The title has to change: the thesis")
    print("   becomes one about 7B teachers, not about the interface.")
else:
    print("=> Intermediate effect. Report both factors as contributors, without")
    print("   attributing the shortfall predominantly to either.")

[15:02:58 + 28.6 min] start: generative


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

[15:04:02 + 29.7 min]   generative 8/695  ETA 26 min
[15:07:12 + 32.8 min]   generative 88/695  ETA 24 min
[15:10:06 + 35.7 min]   generative 168/695  ETA 20 min
[15:12:53 + 38.5 min]   generative 248/695  ETA 17 min
[15:16:45 + 42.4 min]   generative 328/695  ETA 15 min
[15:19:29 + 45.1 min]   generative 408/695  ETA 11 min
[15:22:04 + 47.7 min]   generative 488/695  ETA 8 min
[15:25:04 + 50.7 min]   generative 568/695  ETA 5 min
[15:28:00 + 53.6 min]   generative 648/695  ETA 2 min
[15:29:46 + 55.4 min] saved: generative

--- amostra da saída bruta ---
Subjetivo Paciente <PATIENT>Carlos</PATIENT/> <PATIENT>Menezes</PATIENT/> , <AGE>12</AGE/> anos , <PROFESSION>estudante</PROFESSION/> , apresenta-se com o seguinte histórico : # # # DOT PS # # # <PATIENT>Carlos</PATIENT/> apresenta diagnóstico firmado desde <AGE>3</AGE/> anos , em acompanhamento multidisciplinar . Respostas pouco eficazes a intervenções comportamentais e farmacológicas previamente tentadas . Paciente internado após epi

---
## 5. Task B — ten seeds



In [ ]:
def run_seeds10():
    out = load("seeds10_partial") or {"baseline": [], "kd": []}
    prev = None
    if (P2/"final_runs.json").exists():
        prev = json.loads((P2/"final_runs.json").read_text())
        for cond in ("baseline","kd"):
            for r in prev.get(cond, []):
                if not any(x["seed"]==r["seed"] for x in out[cond]): out[cond].append(r)
        log(f"reused {len(out['baseline'])} seeds from phase 2")
    for seed in CFG.seeds10:
        for cond in ("baseline","kd"):
            if any(r["seed"]==seed for r in out[cond]): continue
            m = train_one(seed, soft=(SOFT if cond=="kd" else None), tag=f"{cond}{seed}")
            yt, yp = predict(m, CORPUS_EVAL["test"]); free(m)
            met = metrics(yt, yp)
            (RUN/"preds"/f"{cond}_{seed}.json").write_text(json.dumps(yp))
            if not (RUN/"preds"/"gold.json").exists():
                (RUN/"preds"/"gold.json").write_text(json.dumps(yt))
            out[cond].append({"seed": seed, **slim(met)})
            save("seeds10_partial", out)
            log(f"{cond:8s} seed {seed:3d}: macro {met['macro_f1']:.4f} micro {met['micro_f1']:.4f}")
    return out

FINAL = task("seeds10", run_seeds10)
for c in FINAL:
    ma=[r["macro_f1"] for r in FINAL[c]]; mi=[r["micro_f1"] for r in FINAL[c]]
    print(f"{c:9s} n={len(ma):2d}  macro {np.mean(ma):.4f} ± {np.std(ma,ddof=1):.4f}"
          f"   micro {np.mean(mi):.4f} ± {np.std(mi,ddof=1):.4f}")

[15:29:46 + 55.4 min] start: seeds10
[15:29:47 + 55.4 min] reaproveitadas 5 sementes da fase 2


pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  438MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  438MB            

...

{'train_runtime': '88.04', 'train_samples_per_second': '127.3', 'train_steps_per_second': '8.008', 'train_loss': '0.403', 'epoch': '5'}
[15:41:07 + 66.7 min] saved: seeds10_partial
[15:41:07 + 66.7 min] kd       seed 202: macro 0.8073 micro 0.9519


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

{'train_runtime': '71.46', 'train_samples_per_second': '156.8', 'train_steps_per_second': '9.866', 'train_loss': '0.1427', 'epoch': '5'}
[15:42:23 + 68.0 min] saved: seeds10_partial
[15:42:23 + 68.0 min] baseline seed 314: macro 0.8081 micro 0.9528


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

{'train_runtime': '88.4', 'train_samples_per_second': '126.8', 'train_steps_per_second': '7.975', 'train_loss': '0.4096', 'epoch': '5'}
[15:43:56 + 69.6 min] saved: seeds10_partial
[15:43:56 + 69.6 min] kd       seed 314: macro 0.8006 micro 0.9504
[15:43:56 + 69.6 min] saved: seeds10
baseline  n=10  macro 0.8083 ± 0.0032   micro 0.9523 ± 0.0011
kd        n=10  macro 0.8058 ± 0.0030   micro 0.9518 ± 0.0008


In [ ]:
from scipy import stats
common = sorted({r["seed"] for r in FINAL["baseline"]} & {r["seed"] for r in FINAL["kd"]})
bb = [next(r["macro_f1"] for r in FINAL["baseline"] if r["seed"]==s) for s in common]
kk = [next(r["macro_f1"] for r in FINAL["kd"]       if r["seed"]==s) for s in common]
dd = np.array(kk) - np.array(bb); n = len(dd)
t_stat, p_val = stats.ttest_rel(kk, bb)
tcrit = stats.t.ppf(0.975, n-1); se = dd.std(ddof=1)/np.sqrt(n)
ci = (dd.mean()-tcrit*se, dd.mean()+tcrit*se)
dz = dd.mean()/dd.std(ddof=1)
nc = abs(dz)*np.sqrt(n)
power = 1 - stats.nct.cdf(tcrit, n-1, nc) + stats.nct.cdf(-tcrit, n-1, nc)
STATS = {"n_seeds": n, "seeds": common, "mean_delta_pp": float(100*dd.mean()),
         "sd_pp": float(100*dd.std(ddof=1)), "ci_pp": [float(100*ci[0]), float(100*ci[1])],
         "t": float(t_stat), "p": float(p_val), "cohens_d": float(dz), "power": float(power)}
save("statistics10", STATS)
print(f"n = {n} paired seeds")
print(f"mean delta {STATS['mean_delta_pp']:+.2f} pp   95% CI "
      f"[{STATS['ci_pp'][0]:+.2f}, {STATS['ci_pp'][1]:+.2f}]")
print(f"t = {STATS['t']:.3f}   p = {STATS['p']:.4f}   d = {STATS['cohens_d']:.2f}   "
      f"power = {100*STATS['power']:.0f}%")

[15:43:56 + 69.6 min] saved: statistics10
n = 10 sementes pareadas
delta médio -0.25 pp   IC95% [-0.50, -0.01]
t = -2.343   p = 0.0438   d = -0.74   poder = 55%


---
## 6. Consolidation and download

In [ ]:
OUT = {"generated_at": time.strftime("%Y-%m-%dT%H:%M:%S"),
       "qwen_generative": {k:v for k,v in GEN.items() if k not in ("per_type","raw_samples")},
       "qwen_generative_per_type": GEN.get("per_type"),
       "qwen_constrained_reference": {"macro_f1":0.1886,"micro_f1":0.3244,"micro_recall":0.2836},
       "interface_gap_pp": float(100*(GEN["micro_f1"]-0.3244)),
       "final_runs_10seeds": FINAL, "statistics10": STATS}
save("results_phase4", OUT)

print("="*66); print("NUMBERS FOR THE MANUSCRIPT"); print("="*66)
print(f"Qwen generative    macro {GEN['macro_f1']:.4f}  micro {GEN['micro_f1']:.4f}")
print(f"Qwen constrained  macro 0.1886          micro 0.3244        (phase 2)")
print(f"interface gap     {OUT['interface_gap_pp']:+.2f} pp micro, same model, same metric")
print(f"GPT-4o generative micro 0.9089                              (Schiezaro et al.)")
print(f"BERTimbau         micro {np.mean([r['micro_f1'] for r in FINAL['baseline']]):.4f}"
      f"  (n={len(FINAL['baseline'])})")
print(f"delta KD          {STATS['mean_delta_pp']:+.2f} pp  95% CI [{STATS['ci_pp'][0]:+.2f},"
      f" {STATS['ci_pp'][1]:+.2f}]  p={STATS['p']:.4f}  power={100*STATS['power']:.0f}%")

z = Path(shutil.make_archive(f"/content/phase4_{time.strftime('%Y%m%d_%H%M%S')}","zip",RUN))
shutil.copy2(z, DRIVE/z.name); print(f"\npackage: {DRIVE/z.name}")
try:
    from google.colab import files; files.download(str(z))
except Exception: pass

[15:43:56 + 69.6 min] saved: results_phase4
NÚMEROS PARA O MANUSCRITO
Qwen generativo   macro 0.4328  micro 0.6556
Qwen restrito     macro 0.1886          micro 0.3244        (fase 2)
lacuna interface  +33.12 pp micro, mesmo modelo, mesma métrica
GPT-4o generativo micro 0.9089                              (Schiezaro et al.)
BERTimbau         micro 0.9523  (n=10)
delta KD          -0.25 pp  IC [-0.50, -0.01]  p=0.0438  poder=55%

pacote: /content/drive/MyDrive/anonymed/phase4_20260815_154356.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>